# Kaggle eval runner (thin)

Runs `src.evaluate` with vLLM on one T4. Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Optional secret: `HF_TOKEN` (Add-ons → Secrets).

The defaults below are the **Step 0 smoke test**: untrained Qwen2.5-3B-Instruct on dev, written to the ignored `results/raw/`. This checks the environment only; the base model is not a study condition. For a real eval, change `MODEL` / `RUN_NAME`. `--split test` is refused unless `--final` is passed.

In [ ]:
# --- run settings ---
REPO = "https://github.com/twiggytom/tiny-gambits.git"
COMMIT = "main"            # pin to a commit hash for reported runs
MODEL = "Qwen/Qwen2.5-3B-Instruct"
SPLIT = "dev"
RUN_NAME = "raw/smoke_3b_base"
GPU = "0"                  # one eval job per T4; use "1" for a second job in parallel
SEED = 0
LIMIT = None               # e.g. 20 for a quick check

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working
!rm -rf tiny-gambits && git clone -q {REPO} tiny-gambits
%cd /kaggle/working/tiny-gambits
!git checkout -q {COMMIT} && git log -1 --format='%H %s'

In [ ]:
# Kaggle ships torch/pandas/numpy; install only what the harness needs on top.
# The first successful run's freeze is written next to the results and becomes requirements-gpu.txt.
!pip install -q chess==1.11.2 PyYAML vllm
!pip freeze > /kaggle/working/pip_freeze.txt
!grep -iE '^(torch|vllm|transformers|chess|pandas|numpy|pyarrow|huggingface.hub)==' /kaggle/working/pip_freeze.txt

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret (fine for public models/datasets):", type(e).__name__)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU

In [ ]:
limit = f"--limit {LIMIT}" if LIMIT else ""
!python -m src.evaluate --config configs/eval.yaml --model {MODEL} --backend vllm --split {SPLIT} --run-name {RUN_NAME} --seed {SEED} {limit}

In [ ]:
import json, shutil
out = f"results/{RUN_NAME}/{SPLIT}"
shutil.copy("/kaggle/working/pip_freeze.txt", f"{out}/pip_freeze.txt")
info = json.load(open(f"{out}/run_info.json"))
print(json.dumps({k: info[k] for k in ["gpu", "dtype", "seconds", "versions"]}, indent=2))
m = json.load(open(f"{out}/metrics.json"))
print(json.dumps({k: m[k] for k in ["n_puzzles", "first_move_acc", "full_solution_acc", "illegal_rate", "unparseable_rate"]}, indent=2))
!head -c 1500 {out}/puzzles.jsonl

In [ ]:
# vLLM vs HF parity check on the first 20 puzzles (same model, greedy). Moves should match;
# rare fp16 kernel differences are possible and get logged, not hidden.
import pandas as pd
!python -m src.evaluate --config configs/eval.yaml --model {MODEL} --backend hf --split {SPLIT} --limit 20 --run-name raw/parity_hf --seed {SEED}
v = pd.read_json(f"results/{RUN_NAME}/{SPLIT}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
h = pd.read_json(f"results/raw/parity_hf/{SPLIT}/puzzles.jsonl", lines=True, dtype={"PuzzleId": str}).set_index("PuzzleId")
cols = ["model_outputs", "outcome"]
print("compared columns:", cols)
diff = [pid for pid in h.index if any(str(v.loc[pid, c]) != str(h.loc[pid, c]) for c in cols)]
print(f"{len(h) - len(diff)}/{len(h)} puzzles identical; differing: {diff}")

In [ ]:
# Zip results so they can be downloaded from the Output tab and committed locally.
shutil.make_archive(f"/kaggle/working/{RUN_NAME.replace('/', '_')}_{SPLIT}", "zip", out)